<a href="https://colab.research.google.com/github/maricmello/Natural-Language-Processing/blob/main/Embbeding/03_Embbedings_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nome: Mariana Costa de Mello

# Fixando a seed

In [ ]:
import random
import torch
import numpy as np

In [ ]:
def set_seeds():
  random.seed(123)
  np.random.seed(123)
  torch.manual_seed(123)
  torch.cuda.manual_seed(123)
set_seeds()

## Preparando Dados

Primeiro, fazemos download do dataset:

In [ ]:
!curl -L -o aclImdb.tgz http://files.fast.ai/data/aclImdb.tgz
!tar -xzf aclImdb.tgz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   167    0   167    0     0   4857      0 --:--:-- --:--:-- --:--:--  4911
100  139M  100  139M    0     0  64.0M      0  0:00:02  0:00:02 --:--:--  115M


## Carregando o dataset

Criaremos uma divisão de treino (80%) e validação (20%) artificialmente.

Nota: Evitar de olhar ao máximo o dataset de teste para não ficar enviseado no que será testado. Em aplicações reais, o dataset de teste só estará disponível no futuro, ou seja, é quando o usuário começa a testar o seu produto.

In [ ]:
import os
import random


def load_texts(folder):
    texts = []
    for path in os.listdir(folder):
        with open(os.path.join(folder, path)) as f:
            texts.append(f.read())
    return texts

x_train_pos = load_texts('aclImdb/train/pos')
x_train_neg = load_texts('aclImdb/train/neg')
x_test_pos = load_texts('aclImdb/test/pos')
x_test_neg = load_texts('aclImdb/test/neg')

x_train = x_train_pos + x_train_neg
x_test = x_test_pos + x_test_neg
y_train = [True] * len(x_train_pos) + [False] * len(x_train_neg)
y_test = [True] * len(x_test_pos) + [False] * len(x_test_neg)

# Embaralhamos o treino para depois fazermos a divisão treino/valid.
c = list(zip(x_train, y_train))
random.shuffle(c)
x_train, y_train = zip(*c)

n_train = int(0.8 * len(x_train))

x_valid = x_train[n_train:]
y_valid = y_train[n_train:]
x_train = x_train[:n_train]
y_train = y_train[:n_train]

print(len(x_train), 'amostras de treino.')
print(len(x_valid), 'amostras de desenvolvimento.')
print(len(x_test), 'amostras de teste.')

print('3 primeiras amostras treino:')
for x, y in zip(x_train[:3], y_train[:3]):
    print(y, x[:100])

print('3 últimas amostras treino:')
for x, y in zip(x_train[-3:], y_train[-3:]):
    print(y, x[:100])

print('3 primeiras amostras validação:')
for x, y in zip(x_valid[:3], y_test[:3]):
    print(y, x[:100])

print('3 últimas amostras validação:')
for x, y in zip(x_valid[-3:], y_valid[-3:]):
    print(y, x[:100])

20000 amostras de treino.
5000 amostras de desenvolvimento.
25000 amostras de teste.
3 primeiras amostras treino:
False Recap: Full moon. A creature, a huge werewolf, is on the hunt. Not for flesh, not for blood (not tha
False I love watching Jerry as much as the rest of the world, but this poor excuse for a soft-core porno f
True When I was a kid it was Lex Barker's time as Tarzan. I often heard from older people that Johnny Wei
3 últimas amostras treino:
False This film was very well advertised. I am an avid movie goer and have seen previews for this movie fo
True I have seen this movie more than several times, on TV. I ALWAYS watch it again...NEVER turning the c
True Trick or Treat, Quickie Review This zany romp of a film revolves around the 80's culture of Heavy Me
3 primeiras amostras validação:
True Most horror movies are in fact horrible movies. They get to be same ol'-same ol'. Same ol' pack ever
True My Brother And I Have Pokemon 4Ever On DVD. We Watched It Like A Couple Of Ti

In [ ]:
sum([len(item.split()) for item in x_train])

4672536

# Imports

In [ ]:
%matplotlib inline
import numpy as np
from bs4 import BeautifulSoup
import torch
from torch.utils.data import DataLoader

import re
from collections import Counter, OrderedDict
import numpy as np

import matplotlib.pyplot as plt



In [ ]:
if torch.cuda.is_available():
   dev = "cuda:0"
   print(torch. cuda. get_device_name(dev))
else:
   dev = "cpu"
print(dev)
device = torch.device(dev)

Tesla T4
cuda:0


# Shuffle DS

In [ ]:
c = list(zip(x_train, y_train))
d = list(zip(x_valid, y_valid))

random.shuffle(c)
x_valid, y_valid = zip(*d)
x_train, y_train = zip(*c)
len(x_train)

20000

# Tokenizer

In [ ]:
def tokenizer(sentence):
    #Beautiful soup to remove tags of sentences
    removeTags = BeautifulSoup(sentence, 'html.parser')

    # Getting list of words and remove numbers and ponctuation with regex
    removeNotWords = re.sub('\W+',' ', removeTags.text)
    regex = re.compile('\w+|[^\w\s]+')
    re_split = regex.findall(removeNotWords.lower())

    re_split_without_numbers = [word for word in re_split if not word.isnumeric()]
    return re_split_without_numbers
x_train_token = tokenizer(' '.join(x_train))

<>:6: SyntaxWarning: invalid escape sequence '\W'
<>:7: SyntaxWarning: invalid escape sequence '\w'
<>:6: SyntaxWarning: invalid escape sequence '\W'
<>:7: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_679/2985889524.py:6: SyntaxWarning: invalid escape sequence '\W'
  removeNotWords = re.sub('\W+',' ', removeTags.text)
/tmp/ipykernel_679/2985889524.py:7: SyntaxWarning: invalid escape sequence '\w'
  regex = re.compile('\w+|[^\w\s]+')


# Vocabulary

In [ ]:
# Size Vocabulary
size_vocab = 10000

# Count tokens
counterTokens = dict(Counter(x_train_token).most_common(size_vocab))

# Count tokens
counterTokens = Counter(x_train_token)

# Define vocabulario sem depender do torchtext (biblioteca descontinuada e que
# costuma dar conflito de versao com o torch em ambientes como o Colab).
# A classe abaixo reproduz o comportamento que usariamos do torchtext.vocab.vocab:
# - vocab_train[token] -> retorna o indice do token (ou o indice padrao, se desconhecido)
# - len(vocab_train) -> numero de tokens no vocabulario
# - set_default_index(idx) -> define o indice usado para tokens fora do vocabulario
class SimpleVocab:
    def __init__(self, counter, min_freq=1):
        self.stoi = {}
        idx = 0
        for word, freq in counter.items():
            if freq >= min_freq:
                self.stoi[word] = idx
                idx += 1
        self.default_index = None

    def set_default_index(self, idx):
        self.default_index = idx

    def __getitem__(self, token):
        return self.stoi.get(token, self.default_index)

    def __len__(self):
        return len(self.stoi)


vocab_train = SimpleVocab(counterTokens, min_freq=1)
vocab_train.set_default_index(len(vocab_train))

print('len of train vocabulary is',len(counterTokens))

bow_pipeline = lambda x: [vocab_train[token] for token in tokenizer(x)]

len of train vocabulary is 68819


In [ ]:
print(bow_pipeline('This movie$#@ &*() is, amazing! fr33'))
bow_pipeline('Thats something Thats something !$ This movie$#@ &*() is, amazing!')

[0, 6, 94, 1621, 68819]


[2948, 163, 2948, 163, 0, 6, 94, 1621]

# Padding parameters

In [ ]:
max_pad = 200
pad_idx = len(vocab_train)+1

# Dataset

In [ ]:
class Ex3_ds(torch.utils.data.Dataset):

    def __init__(self, x, y, mode):
        self.x = x
        self.y = y
        self.mode = mode

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):

        y_label = int(self.y[index])
        token_idx = bow_pipeline(self.x[index])
        token_idx = token_idx[:max_pad]

        if self.mode == 'bow_hist':
            all_tokens = torch.zeros(len(vocab_train)+1)
            for idx in token_idx:
                all_tokens[idx] += 1

        elif self.mode == 'emb':
            all_tokens = torch.full((max_pad,), pad_idx, dtype=torch.long)
            all_tokens[:len(token_idx)] = torch.tensor(token_idx, dtype=torch.long)

        return all_tokens, y_label

Creating weights (W) and bias 2nd layer

For linear/embedding + relu + linear implementation, didn't get same results.

Needed to clone weights of bow_hist model and then .cat with zeros for padding.

# Models

In [ ]:
class Ex3_model(torch.nn.Module):
    def __init__(self, mode, input, hidden):
        super(Ex3_model, self).__init__()
        self.mode = mode
        if mode == 'bow_hist':
            self.fst_layer = torch.nn.Linear(input, hidden, bias=False)
        elif mode == 'emb':
            self.fst_layer = torch.nn.Embedding(input, hidden, padding_idx=pad_idx)
        else:
            print("Invalid mode")

        self.relu = torch.nn.ReLU()
        self.snd_linear_layer = torch.nn.Linear(hidden, 2, bias=False)

    def forward(self, x):
        if self.mode == 'bow_hist':
            x = self.fst_layer(x)
        elif self.mode == 'emb':
            x = self.fst_layer(x)
            x = x.sum(dim=1)

        x = self.relu(x)
        x = self.snd_linear_layer(x)
        return x

# Summary models

In [ ]:
# Model BoW Freq
ex3_model = Ex3_model('bow_hist',len(vocab_train)+1, 128)
ex3_model.to(device)
#print(ex3_model.fst_layer.weight)
#print(ex3_model.fst_layer.weight.shape)

# Summary Model BoW Freq
print(ex3_model)

# Model Embedding
ex3_model_emb = Ex3_model('emb',len(vocab_train)+2, 128)
ex3_model_emb.to(device)
#print(ex3_model_emb.fst_layer.weight)
#print(ex3_model_emb.fst_layer.weight.shape)

# Summary Model Embedding
print(ex3_model_emb)


Ex3_model(
  (fst_layer): Linear(in_features=68820, out_features=128, bias=False)
  (relu): ReLU()
  (snd_linear_layer): Linear(in_features=128, out_features=2, bias=False)
)
Ex3_model(
  (fst_layer): Embedding(68821, 128, padding_idx=68820)
  (relu): ReLU()
  (snd_linear_layer): Linear(in_features=128, out_features=2, bias=False)
)


# Train loop

In [ ]:
n_epochs = 5
learningRate = 0.0001

# CrossEntropyLoss as loss function
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
def train_loop(dataloader_train, dataloader_val, hyperparameters, model):
    train_loss_a = []
    val_loss_a = []
    total_acc_a = []

    optimizer = torch.optim.Adam(model.parameters(), lr=hyperparameters['learning_rate'])
    min_val_loss = 10e9
    best_epoch = 0

    for epoch in range(hyperparameters['n_epochs']):
        train_loss = 0
        model.train()
        for x_train, y_train in dataloader_train:
            x_train = x_train.to(device)
            y_train = y_train.to(device)

            outputs = model(x_train)
            batch_loss = criterion(outputs, y_train)

            optimizer.zero_grad()
            batch_loss.backward()
            optimizer.step()
            train_loss += batch_loss.item()

        train_loss = train_loss / len(dataloader_train.dataset)
        train_loss_a.append(train_loss)

        total_loss = 0
        total_acc = 0
        model.eval()
        with torch.no_grad():
            for x_val, y_val in dataloader_val:
                x_val = x_val.to(device)
                y_val = y_val.to(device)

                outputs = model(x_val)
                batch_loss = criterion(outputs, y_val)
                preds = outputs.argmax(dim=1)

                batch_acc = (preds == y_val).sum()
                total_loss += batch_loss
                total_acc += batch_acc

        val_loss = total_loss / len(dataloader_val.dataset)
        val_loss_a.append(val_loss)
        total_acc_a.append(total_acc / len(dataloader_val.dataset))

        print(f'Model: {hyperparameters["mode"]}, Epoch: {epoch+1}/{hyperparameters["n_epochs"]} - train_loss: {train_loss} - val_loss: {val_loss} - acc: {total_acc / len(dataloader_val.dataset)*100} %')

        if val_loss < min_val_loss:
            torch.save(model.state_dict(), 'best_model.pt')
            min_val_loss = val_loss
            best_epoch = epoch
            print(f'Model: {hyperparameters["mode"]} - best model in epoch: {best_epoch+1}')

    return train_loss_a, val_loss_a, total_acc_a

In [ ]:
# Function to test acc
def predict(model, inputs):
    outputs = model(inputs)
    _, predicts = torch.max(outputs, 1)
    return predicts



# List to Dict

In [ ]:
# Transform list to dict
x_train = {num: i for num, i in enumerate(x_train)}
y_train = {num: i for num, i in enumerate(y_train)}
x_valid = {num: i for num, i in enumerate(x_valid)}
y_valid = {num: i for num, i in enumerate(y_valid)}
x_test = {num: i for num, i in enumerate(x_test)}
y_test = {num: i for num, i in enumerate(y_test)}

# BoW Hist sem Bias

In [ ]:
set_seeds()

In [ ]:
# Defining model
model_bow = Ex3_model('bow_hist', len(vocab_train)+1, 128)

# Saving weights list to use in embedding model
bow_weights = [model_bow.fst_layer.weight.clone(), model_bow.snd_linear_layer.weight.clone()]

model_bow.to(device)
print(f'bow_hist model - Weights shape 1st Layer: {model_bow.fst_layer.weight.shape}')
print(f'bow_hist model - Weights shape 2nd Layer: {model_bow.snd_linear_layer.weight.shape}')
#print(model_bow.fst_layer.weight)
#print(model_bow.snd_linear_layer.weight)

bow_hist model - Weights shape 1st Layer: torch.Size([128, 68820])
bow_hist model - Weights shape 2nd Layer: torch.Size([2, 128])


In [ ]:

hyperparameters = { "mode": "bow_hist",
          "learning_rate": 1e-4,
          "n_epochs": 5,
          "batch_size": 50,
          "hidden_size": 128 }

train_ds = Ex3_ds(x_train, y_train, hyperparameters['mode'])
val_ds = Ex3_ds(x_valid, y_valid, hyperparameters['mode'])

set_seeds()
dataloader_train = DataLoader(train_ds, batch_size=hyperparameters['batch_size'], shuffle=True)
dataloader_val = DataLoader(val_ds, batch_size=hyperparameters['batch_size'], shuffle=False)


In [ ]:
train_loss_bow, val_loss_bow, acc_bow = train_loop(dataloader_train, dataloader_val, hyperparameters, model_bow)

Model: bow_hist, Epoch: 1/5 - train_loss: 0.009469416003674269 - val_loss: 0.006991395261138678 - acc: 86.47999572753906 %
Model: bow_hist - best model in epoch: 1
Model: bow_hist, Epoch: 2/5 - train_loss: 0.005148682027310133 - val_loss: 0.006043595727533102 - acc: 87.89999389648438 %
Model: bow_hist - best model in epoch: 2
Model: bow_hist, Epoch: 3/5 - train_loss: 0.0034558037985116244 - val_loss: 0.005892365705221891 - acc: 87.75999450683594 %
Model: bow_hist - best model in epoch: 3
Model: bow_hist, Epoch: 4/5 - train_loss: 0.002415327846631408 - val_loss: 0.005995358340442181 - acc: 87.63999938964844 %
Model: bow_hist, Epoch: 5/5 - train_loss: 0.0017039538664743305 - val_loss: 0.006294285412877798 - acc: 87.47999572753906 %


In [ ]:
del train_ds
del val_ds
del dataloader_train
del dataloader_val

In [ ]:
test_ds = Ex3_ds(x_test, y_test, hyperparameters['mode'])
dataloader_test = DataLoader(test_ds, batch_size=hyperparameters['batch_size'], shuffle=False)
total_acc = 0
with torch.no_grad():
  for x_t, y_t in dataloader_test:
    x_t = x_t.to(device)
    #print(x_t.shape)
    y_t = y_t.to(device)
    #print(y_t.shape)
    outputs = model_bow(x_t)
    #print(outputs.shape)

    #preds = outputs > 0.5

    preds = outputs.argmax(dim=1)
    #print(preds.shape)

    # test acc
    batch_acc = (preds == y_t).sum()
    total_acc += batch_acc
  test_acc = total_acc / len(dataloader_test.dataset)

  print(f"BoW Hist Acc: {test_acc*100} %")

BoW Hist Acc: 85.43599700927734 %


In [ ]:
del test_ds
del dataloader_test


# Embedding

In [ ]:
set_seeds()

In [ ]:
zeros = torch.zeros(128, 1)

pad_weight = torch.cat([bow_weights[0], zeros], dim=1)
emb_weights = OrderedDict([
    ('fst_layer.weight',  pad_weight.T),
    ('snd_linear_layer.weight',  bow_weights[1])])

In [ ]:
# Defining model
ex3_model_emb = Ex3_model('emb',len(vocab_train)+2, 128)

# Loading weights
ex3_model_emb.load_state_dict(emb_weights)

print(f'emb model - Weights shape 1st Layer: {ex3_model_emb.fst_layer.weight.shape}')
print(f'emb model - Weights shape 2nd Layer: {ex3_model_emb.snd_linear_layer.weight.shape}')

emb model - Weights shape 1st Layer: torch.Size([68821, 128])
emb model - Weights shape 2nd Layer: torch.Size([2, 128])


In [ ]:

hyperparameters = { "mode": "emb",
          "learning_rate": 1e-4,
          "n_epochs": 5,
          "batch_size": 50,
          "hidden_size": 128 }

ex3_model_emb.to(device)

train_ds = Ex3_ds(x_train, y_train, hyperparameters['mode'])
val_ds = Ex3_ds(x_valid, y_valid, hyperparameters['mode'])

set_seeds()
dataloader_train = DataLoader(train_ds, batch_size=hyperparameters['batch_size'], shuffle=True)
dataloader_val = DataLoader(val_ds, batch_size=hyperparameters['batch_size'], shuffle=False)
train_loss_emb, val_loss_emb, acc_emb = train_loop(dataloader_train, dataloader_val, hyperparameters, ex3_model_emb)

Model: emb, Epoch: 1/5 - train_loss: 0.009469415976852179 - val_loss: 0.006991395261138678 - acc: 86.47999572753906 %
Model: emb - best model in epoch: 1
Model: emb, Epoch: 2/5 - train_loss: 0.005148682041466236 - val_loss: 0.006043596658855677 - acc: 87.89999389648438 %
Model: emb - best model in epoch: 2
Model: emb, Epoch: 3/5 - train_loss: 0.0034558038087561726 - val_loss: 0.005892365705221891 - acc: 87.75999450683594 %
Model: emb - best model in epoch: 3
Model: emb, Epoch: 4/5 - train_loss: 0.0024153278583660723 - val_loss: 0.005995358340442181 - acc: 87.63999938964844 %
Model: emb, Epoch: 5/5 - train_loss: 0.001703953878208995 - val_loss: 0.006294285412877798 - acc: 87.47999572753906 %


In [ ]:
del train_ds
del val_ds
del dataloader_train
del dataloader_val

In [ ]:
test_ds = Ex3_ds(x_test, y_test, hyperparameters['mode'])
dataloader_test = DataLoader(test_ds, batch_size=hyperparameters['batch_size'], shuffle=False)
total_acc = 0
with torch.no_grad():
  for x_t, y_t in dataloader_test:
    x_t = x_t.to(device)
    #print(x_t.shape)
    y_t = y_t.to(device)
    #print(y_t.shape)
    outputs = ex3_model_emb(x_t)
    #print(outputs.shape)

    #preds = outputs > 0.5

    preds = outputs.argmax(dim=1)
    #print(preds.shape)

    # test acc
    batch_acc = (preds == y_t).sum()
    total_acc += batch_acc
  test_acc = total_acc / len(dataloader_test.dataset)

  print(f"Embedding Acc: {test_acc*100} %")

Embedding Acc: 85.43599700927734 %


In [ ]:
del test_ds
del dataloader_test

#  Verify if BoW Freq. and Embedding models had same results

In [ ]:
print(f"Train loss BoW Freq. {train_loss_bow}")
print(f"Train loss Embedding {train_loss_emb}")

print(f"Val loss BoW Freq. {val_loss_bow}")
print(f"Val loss Embedding {val_loss_emb}")

print(f"Acc BoW Freq. {acc_bow}")
print(f"Acc Embedding {acc_emb}")


Train loss BoW Freq. [0.009469416003674269, 0.005148682027310133, 0.0034558037985116244, 0.002415327846631408, 0.0017039538664743305]
Train loss Embedding [0.009469415976852179, 0.005148682041466236, 0.0034558038087561726, 0.0024153278583660723, 0.001703953878208995]
Val loss BoW Freq. [tensor(0.0070, device='cuda:0'), tensor(0.0060, device='cuda:0'), tensor(0.0059, device='cuda:0'), tensor(0.0060, device='cuda:0'), tensor(0.0063, device='cuda:0')]
Val loss Embedding [tensor(0.0070, device='cuda:0'), tensor(0.0060, device='cuda:0'), tensor(0.0059, device='cuda:0'), tensor(0.0060, device='cuda:0'), tensor(0.0063, device='cuda:0')]
Acc BoW Freq. [tensor(0.8648, device='cuda:0'), tensor(0.8790, device='cuda:0'), tensor(0.8776, device='cuda:0'), tensor(0.8764, device='cuda:0'), tensor(0.8748, device='cuda:0')]
Acc Embedding [tensor(0.8648, device='cuda:0'), tensor(0.8790, device='cuda:0'), tensor(0.8776, device='cuda:0'), tensor(0.8764, device='cuda:0'), tensor(0.8748, device='cuda:0')]


In [ ]:
assert torch.allclose(torch.tensor(train_loss_bow), torch.tensor(train_loss_emb), rtol=1e-05, atol=1e-08) and torch.allclose(torch.tensor(val_loss_bow), torch.tensor(val_loss_emb), rtol=1e-05, atol=1e-08) and torch.allclose(torch.tensor(acc_bow), torch.tensor(acc_emb), rtol=1e-05, atol=1e-08)

# Comentários

Equivalência numérica entre os modelos
- Sem bias na primeira camada: garante que os cálculos dos modos bow_hist e emb sejam equivalentes.
- Pesos compartilhados: os mesmos pesos são usados nos dois modelos, com a matriz ajustada para o embedding.
- Mesma seed: garante que os dois modelos recebam os mesmos mini-batches durante o treinamento.
- Mesmo truncamento (max_pad): ambos utilizam exatamente os mesmos tokens de cada documento.
- torch.allclose: confirma que as perdas de treino, validação e acurácia são praticamente iguais.
- Pequenas diferenças: podem ocorrer apenas por arredondamentos de ponto flutuante durante os cálculos.

Utilização do Claude para auxilio em dúvidas do código e resumo dos comentários.